# Gym-Locker Training on Kaggle

This notebook trains the Gym-Locker RL agent on Kaggle with GPU acceleration.

## Setup Instructions

1. Upload this notebook to Kaggle
2. Enable GPU (Settings > Accelerator > GPU)
3. Add the gym-locker repository as a dataset or clone from GitHub
4. Run all cells
5. Download trained models from output

## 1. Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install gymnasium pygame opencv-python stable-baselines3 sb3-contrib simple-pid tensorboard

## 2. Clone Repository (if not added as dataset)

In [ ]:
# Clone repository from GitHub
# Uncomment if repository is not already available
# !git clone https://github.com/YOUR_USERNAME/gym-locker.git
# import sys
# sys.path.append('/kaggle/working/gym-locker')

## 3. Import Libraries

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
from stable_baselines3 import PPO, SAC, TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback

# Add gym-locker to path (adjust path as needed)
sys.path.append('/kaggle/working/gym-locker')

import gym_locker
from gym_locker.agents.networks import get_feature_extractor
from gym_locker.utils.visualization import plot_training_metrics

print("Imports successful!")
print(f"PyTorch GPU available: {os.popen('nvidia-smi').read() != ''}")

## 4. Create Environment

In [ ]:
# Set SDL to use dummy video driver (no display)
os.environ['SDL_VIDEODRIVER'] = 'dummy'

def make_env(state_mode="vector", evader_difficulty=0.5, evader_speed_multiplier=1.5):
    """Create environment without rendering."""
    env = gym.make(
        'LockOn-v0',
        render_mode=None,  # No rendering on Kaggle
        state_mode=state_mode,
        evader_difficulty=evader_difficulty,
        evader_speed_multiplier=evader_speed_multiplier
    )
    return env

# Test environment
test_env = make_env()
obs, info = test_env.reset()
print("Environment created successfully!")
print(f"Observation space: {test_env.observation_space}")
print(f"Action space: {test_env.action_space}")
print(f"Initial observation: {obs}")
test_env.close()

## 5. Training Configuration

In [ ]:
# Configuration
CONFIG = {
    'algorithm': 'PPO',  # Options: PPO, SAC, TD3
    'state_mode': 'vector',  # Options: vector, image
    'architecture': 'simple',  # Options: simple, deep
    'total_timesteps': 500000,  # Increase for better performance
    'evader_difficulty': 0.5,
    'evader_speed_multiplier': 1.5,  # Evader is 1.5x faster than pursuer
    'learning_rate': 3e-4,
    'n_steps': 2048,  # PPO-specific
    'batch_size': 64,
    'checkpoint_freq': 10000,
    'eval_freq': 5000,
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 6. Create Training Environment

In [ ]:
# Create training and evaluation environments
train_env = make_env(CONFIG['state_mode'], CONFIG['evader_difficulty'], CONFIG['evader_speed_multiplier'])
train_env = Monitor(train_env)
train_env = DummyVecEnv([lambda: train_env])

eval_env = make_env(CONFIG['state_mode'], CONFIG['evader_difficulty'], CONFIG['evader_speed_multiplier'])
eval_env = Monitor(eval_env)

print("Training and evaluation environments created!")

## 7. Create Model

In [ ]:
# Get feature extractor
feature_extractor_class = get_feature_extractor(
    CONFIG['state_mode'],
    CONFIG['architecture']
)

# Policy kwargs
policy_kwargs = dict(
    features_extractor_class=feature_extractor_class,
    features_extractor_kwargs=dict(features_dim=64),
)

# Create model
if CONFIG['algorithm'] == 'PPO':
    model = PPO(
        "MlpPolicy" if CONFIG['state_mode'] == "vector" else "CnnPolicy",
        train_env,
        policy_kwargs=policy_kwargs,
        learning_rate=CONFIG['learning_rate'],
        n_steps=CONFIG['n_steps'],
        batch_size=CONFIG['batch_size'],
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        verbose=1,
        tensorboard_log="./tensorboard/"
    )
elif CONFIG['algorithm'] == 'SAC':
    model = SAC(
        "MlpPolicy" if CONFIG['state_mode'] == "vector" else "CnnPolicy",
        train_env,
        policy_kwargs=policy_kwargs,
        learning_rate=CONFIG['learning_rate'],
        buffer_size=100000,
        learning_starts=1000,
        batch_size=256,
        tau=0.005,
        gamma=0.99,
        verbose=1,
        tensorboard_log="./tensorboard/"
    )
elif CONFIG['algorithm'] == 'TD3':
    model = TD3(
        "MlpPolicy" if CONFIG['state_mode'] == "vector" else "CnnPolicy",
        train_env,
        policy_kwargs=policy_kwargs,
        learning_rate=CONFIG['learning_rate'],
        buffer_size=100000,
        learning_starts=1000,
        batch_size=256,
        tau=0.005,
        gamma=0.99,
        verbose=1,
        tensorboard_log="./tensorboard/"
    )

print(f"{CONFIG['algorithm']} model created!")
print(f"Model parameters: {sum(p.numel() for p in model.policy.parameters())}")

## 8. Setup Callbacks

In [ ]:
# Create directories
os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./best_models", exist_ok=True)

# Checkpoint callback
checkpoint_callback = CheckpointCallback(
    save_freq=CONFIG['checkpoint_freq'],
    save_path="./checkpoints/",
    name_prefix=f"{CONFIG['algorithm']}_model"
)

# Evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./best_models/",
    log_path="./eval_logs/",
    eval_freq=CONFIG['eval_freq'],
    deterministic=True,
    render=False
)

print("Callbacks configured!")

## 9. Train Model

In [ ]:
# Train!
print("Starting training...\n")

model.learn(
    total_timesteps=CONFIG['total_timesteps'],
    callback=[checkpoint_callback, eval_callback],
    progress_bar=True
)

print("\nTraining completed!")

## 10. Save Final Model

In [ ]:
# Save final model
final_model_path = f"./final_{CONFIG['algorithm']}_model.zip"
model.save(final_model_path)
print(f"Final model saved to: {final_model_path}")

## 11. Evaluate Model

In [ ]:
# Evaluate final model
n_eval_episodes = 100
episode_rewards = []
episode_lengths = []
lock_times = []

for episode in range(n_eval_episodes):
    obs, _ = eval_env.reset()
    episode_reward = 0
    episode_length = 0
    lock_time = 0
    done = False
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        
        episode_reward += reward
        episode_length += 1
        if info['is_locked']:
            lock_time += 1
        
        done = terminated or truncated
    
    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_length)
    lock_times.append(lock_time)

print("\nEvaluation Results:")
print(f"  Mean Reward: {np.mean(episode_rewards):.2f} ± {np.std(episode_rewards):.2f}")
print(f"  Mean Episode Length: {np.mean(episode_lengths):.2f}")
print(f"  Mean Lock Time: {np.mean(lock_times):.2f}")

## 12. Plot Results

In [ ]:
# Plot evaluation results
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# Rewards
axes[0].hist(episode_rewards, bins=20, edgecolor='black')
axes[0].axvline(np.mean(episode_rewards), color='red', linestyle='--', label='Mean')
axes[0].set_xlabel('Episode Reward')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Episode Rewards')
axes[0].legend()
axes[0].grid(True)

# Episode lengths
axes[1].hist(episode_lengths, bins=20, edgecolor='black')
axes[1].axvline(np.mean(episode_lengths), color='red', linestyle='--', label='Mean')
axes[1].set_xlabel('Episode Length (steps)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Episode Lengths')
axes[1].legend()
axes[1].grid(True)

# Lock times
axes[2].hist(lock_times, bins=20, edgecolor='black')
axes[2].axvline(np.mean(lock_times), color='red', linestyle='--', label='Mean')
axes[2].set_xlabel('Lock-On Time (steps)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Lock-On Times')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150)
plt.show()

print("Results saved to evaluation_results.png")

## 13. Download Models

After running all cells:
1. Go to Output section on the right
2. Download the model files:
   - `final_PPO_model.zip` (or SAC/TD3)
   - Best model from `best_models/` directory
3. Use these models locally for testing with `test.py`

In [ ]:
# List all saved files
print("\nSaved files:")
print("\nFinal model:")
!ls -lh final_*.zip

print("\nBest models:")
!ls -lh best_models/

print("\nCheckpoints:")
!ls -lh checkpoints/

## 14. Cleanup

In [ ]:
# Close environments
train_env.close()
eval_env.close()

print("Training session completed successfully!")